### Setup

In [ ]:
# install dependencies for virtual env
!pip install -r requirements.txt -qqq

In [ ]:
# import librariess
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
#gpu acceleration + declaring model name and layer
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
name = "Qwen/Qwen3-0.6B"
LAYER = 18

#download and load: 
#qwen's tokenizer
tok = AutoTokenizer.from_pretrained(name)

#qwen's model
model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.bfloat16).to(device)

#configure model for inference
model.eval()

#check model config: # of hidden layers and dimensionality of each layer
print(model.config.num_hidden_layers, model.config.hidden_size)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

28 1024


### Basics

In [ ]:
text = "Rent was due in two days and Maria refreshed the bank page again."

#tokenize the above input, maps each token to an ID and returns a tensor of those IDs. 
#these IDs are then moved to the same device the model lives on 
#here, im running this on my mac, so it moves it to the MPS device the model is on. 
ids = tok(text, return_tensors="pt").to(device)

#return the shape of ID tensor created above
print(ids.input_ids.shape)

#return the tokens corresponding to the IDs
print(tok.convert_ids_to_tokens(ids.input_ids[0]))

torch.Size([1, 14])
['Rent', 'Ġwas', 'Ġdue', 'Ġin', 'Ġtwo', 'Ġdays', 'Ġand', 'ĠMaria', 'Ġrefreshed', 'Ġthe', 'Ġbank', 'Ġpage', 'Ġagain', '.']


### Extract Internal States

In [ ]:
#disable gradient computation graph, and feed the token IDs into the model.
with torch.no_grad():
    out = model(**ids, output_hidden_states=True)

#print the number of hidden layers (28+1)
print(len(out.hidden_states))

#print the shape of residual stream of vectors @ the 18th layer
#14 vectors, 1 for each token, each of size 1024 dimensions. 
print(out.hidden_states[LAYER].shape)

29
torch.Size([1, 14, 1024])


### Generate Text from the Model

In [ ]:
def chat(user_msg, max_new_tokens=60, sample=False):
    #initiate conversation. 
    msgs = [{"role": "user", "content": user_msg}]

    #wrap message in markers qwen uses to identify user vs assistant messages.
    #add_generation_prompt=True hints the model to generate a response at the end of the input prompt.
    prompt = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False, enable_thinking=False)

    #tokenize the input, get IDs Tensor. Same as before.
    inp = tok(prompt, return_tensors="pt").to(device)

    #start the autoregressive loop to predict token, append to its stream and repeat.
    #max_new_tokens=60 caps the number of output tokens @ 60. 
    #do_sample=False makes the model deterministic, and it will only output the most likely token. 
    #temperature=0.8 makes the model more creative, plays around with the perplexity of the next token
    gen = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=sample,
                         temperature=0.8 if sample else None)
    
    #extract 1st back-and-forth of the conversation, drop the input prompt tokens, 
    # and decodes the output tokens from IDs to text
    return tok.decode(gen[0][inp.input_ids.shape[1]:], skip_special_tokens=True)

print(chat("Write one sentence about a desperate person."))

A desperate person, trapped in a moment of crisis, searches desperately for hope.


whats next? actual probing/steering! 